In [ ]:
import sys
from pathlib import Path

import os
import sys
from pathlib import Path
import json
import warnings
import random

import torch
from torch.nn import functional as F
import nibabel as nib
import pandas as pd
import numpy as np
import pytorch_lightning as pl
from tqdm import tqdm
from torch.utils.data import DataLoader


from lit_classifier import LitClassifier
from utils import confusion_matrix_to_figure
from transformations import get_transforms
from utils_hparams import args_short_to_long

warnings.filterwarnings("ignore")

seed = 2025
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True  # CuDNN deterministic
torch.backends.cudnn.benchmark = False  # CuDNN benchmark off
torch.random.manual_seed(seed)
pl.seed_everything(seed)

In [ ]:
best_weights_path = '$PATH/RI_anatomical_prior/log/basic_0619/ratio/version_3/checkpoints/epoch=016.ckpt'

def _add_missing_hparams(model):
    hparams_all = {v[0]: v[0] for k, v in args_short_to_long.items()}  # transform to dict param_long_name:default_value
    for param, default_value in hparams_all.items():
        if param not in model.hparams:
            if "lightning_quiet" not in os.environ:
                print(f"Missing hparam '{param}' in stored model. Adding with value {default_value}.")
            model.hparams[param] = default_value
    return model


device = torch.device('cuda:0')

models = []
model = LitClassifier.load_from_checkpoint(best_weights_path)
model = _add_missing_hparams(model)
model.eval().to(device)
models.append(model)
model = models[0] 

subjects_test = model.hparams.subjects_test
labels_test = model.hparams.labels_test
wm_test = model.hparams.wm_test
icv_test = model.hparams.icv_test
ratio_test = model.hparams.ratio_test

tfs_train, tfs_val = get_transforms(model.hparams)

In [ ]:
import torch
import nibabel as nib
import numpy as np

from classifier.lit_classifier import LitClassifier

# ------------------------
# ------------------------
ckpt_path = '$PATH/RI_anatomical_prior/log/basic_0619/ratio/version_3/checkpoints/epoch=016.ckpt'
model = LitClassifier.load_from_checkpoint(ckpt_path)
model.eval()
model.cuda()

file_list = ['case_0001']

for file in file_list:
    nifti_path = f"$PATH/external_cohort/{file}/ratio_clipped_95.nii.gz"
    img = nib.load(nifti_path).get_fdata()   # numpy array (H, W, D)

    x = img[None, ...]
    x = torch.tensor(x, dtype=torch.float32)
    
    ## W aug ##
    x = tfs_val(x)
    x = x.unsqueeze(0)

    ## W/O aug ##

    x = x.cuda()

    with torch.no_grad():
        y_pred = model(x)

    print("Predicted:", y_pred.cpu().numpy())


In [ ]:
import torch
import nibabel as nib
import numpy as np

from classifier.lit_classifier_seg import LitClassifier

# ------------------------
# ------------------------
ckpt_path = '$PATH/RI_anatomical_prior/classifier/RI_BAE/8tt6afi1/checkpoints/epoch=054.ckpt'
model = LitClassifier.load_from_checkpoint(ckpt_path)
model.eval()
model.cuda()

file_list = ['case_0001']

for file in file_list:
    nifti_path = f"$PATH/external_cohort/{file}/ratio_clipped_90.nii.gz"
    img = nib.load(nifti_path).get_fdata()   # numpy array (H, W, D)

    x = img[None, ...]
    x = torch.tensor(x, dtype=torch.float32)
    
    ## W aug ##

    ## W/O aug ##
    x = x.unsqueeze(0)

    x = x.cuda()

    with torch.no_grad():
        y_pred = model(x)

    print("Predicted:", y_pred[-1])
